# Pumpkin-Spice-Season — a quantitative teardown 🔬
### Per-month HAC t-stats on SBUX−SPY · season-vs-off Welch spread + block-bootstrap CI · a 12-window placebo · rotation race · sub-period split

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Pumpkin premium?: Busted](https://img.shields.io/badge/Pumpkin_premium%3F-Busted-8b949e?style=flat-square)

The deep companion to [the notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* We test whether Starbucks' market-relative return concentrates in the Aug–Nov pumpkin-spice window across 400 months and find no signal, a window placebo that an off-thesis quarter wins, a sign-flipping sub-period, and a rotation whose only edge is borrowed single-name beta.

> ⚠️ Not investment advice. SBUX and SPY **total-return** (dividends reinvested), 13-week T-bill (^IRX) the cash leg, monthly 1993-02 → 2026-05, 400 months (Yahoo Finance, daily closes resampled to month-end, grid asserted hole-free). The object of study is the **excess** SBUX − SPY. Offline, every cell falls back to the synthetic NULL and banners the tape. Sources in [`docs/references.md`](../docs/references.md).
>
> 💡 The `💡 In plain words` notes translate each result back into intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (pumpkin_spice_season/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from pumpkin_spice_season import data, strategy as st

# Cache-first real tape; fall back to the synthetic control OFFLINE and banner which tape we show.
try:
    from quantlab import repro
    d = repro.as_of(data.fetch_data())   # cache-first (examples/verify.py --fetch)
    if d.empty:
        raise RuntimeError("cache miss")
    TAPE = "REAL  (SBUX/SPY total-return monthly, 1993-02..2026-05, 400 months, fp 1d50b4e1e153)"
except Exception as e:
    d, _ = data.synthetic_world(psl_premium=0.0, seed=724)  # the NULL synthetic world
    TAPE = f"SYNTHETIC NULL control (offline fallback: {e}) -- NOT the real tape"
print("TAPE:", TAPE)
rf     = d["tbill"]
excess = d["excess"]                             # SBUX - SPY, the object of study
ms     = st.month_stats(excess)
se     = st.season_tstat(excess)
rot    = st.seasonal_rotation(d["sbux"], d["spy"])
rot_net= st.apply_costs(rot, n_trades_per_year=2, cost_bps_one_way=5)
timer  = st.spread_timer(excess, tbill=rf)       # market-neutral, Aug-Nov only
timer_net = st.apply_costs(timer, n_trades_per_year=4, cost_bps_one_way=5)
bh_spy = st.buy_hold(d["spy"]); bh_sbux = st.buy_hold(d["sbux"])


TAPE: REAL  (SBUX/SPY total-return monthly, 1993-02..2026-05, 400 months, fp 1d50b4e1e153)


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **None** | season−off excess spread t = 0.65, 95% CI [−0.84%, +1.92%]; no month \|t\| ≥ 2; PSL window ranks #2 of 12 |
| Tradability | **Mirage** | market-neutral seasonal pair Sharpe 0.27 < always-hold pair 0.35 — the season *dilutes* the edge |
| Pumpkin premium? | **Busted** | −0.69% in 1993–2009 (t = −0.47), only +1.84% in a snooped 2010-on half (t = 2.06) |

> 💡 In plain words: Starbucks beats the market in lots of months. It does **not** do so *especially* in pumpkin-spice season — the vivid calendar story leaves no statistical fingerprint.

## 1 · The claim, steelmanned

- **H₁:** at least one pumpkin-spice-season month (Aug–Nov) has a significantly positive SBUX−SPY excess.
- **H₂:** the season group (Aug–Nov) has a significantly higher mean excess than the off-season — Welch t-test on pooled groups.
- **H₃:** the season-minus-off spread's block-bootstrap 95% CI excludes zero.
- **H₄:** the pumpkin-spice window is the *strongest* four-month window (placebo — it should stand out among all twelve).
- **H₅:** a long-SBUX-in-season / SPY-otherwise rotation beats buy-and-hold *after* accounting for the fact that it's just borrowing SBUX beta, and the pattern is stable across sub-periods.

## 2 · So what? — what rides on each

If H₁–H₅ hold, a marketing calendar prints market-relative alpha and you harvest it with a fixed-month rotation. If they fail, "pumpkin-spice season" is **single-name survivorship dressed as a calendar**: SBUX was a great stock, so *any* slice of its year looks good, and we remember the slice with the cultural story attached.

## 3 · How we'd know — the protocol

One-sample t-stats (naive **and** Newey-West HAC) for each of the 12 calendar months of SBUX−SPY vs 0 (Bonferroni threshold |t| ≈ 3 for α = 0.05/12); a Welch two-sample test for season (Aug–Nov) vs off-season; a circular block-bootstrap (12-month blocks) 95% CI on the spread; a **12-window placebo** sliding a 4-month window around the calendar; the rotation (long SBUX Aug–Nov, SPY otherwise — calendar-known, **no execution lag**) and a market-neutral spread timer, Sharpe in **excess of the T-bill**, gross and net of 5 bp/leg; and a 1993–2009 / 2010-on sub-period split. **Survivorship is on the Signal axis:** SBUX is one hand-picked survivor, so we read the raw-return win with maximum suspicion and lean on the market-relative and placebo tests.

## 4 · The teardown

### 4.1 Per-month excess t-stats, naive and HAC (H₁)

In [2]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print(f'TAPE: {TAPE}\n')
print(f'{"Month":6s}  {"Mean":>8s}  {"t-naive":>8s}  {"t-HAC":>8s}  {"n":>4s}  Signal?')
for m in range(1,13):
    row = ms.loc[m]
    tag = ' (PSL)' if m in [8,9,10,11] else ''
    sig = '|t|>=3 (Bonf.)' if abs(row['tstat_hac'])>=3 else ('|t|>=2' if abs(row['tstat_hac'])>=2 else 'noise')
    print(f'{month_names[m-1]+tag:6s}  {row["mean"]*100:+7.2f}%  {row["tstat"]:+8.2f}  '
          f'{row["tstat_hac"]:+8.2f}  {int(row["n"]):4d}  {sig}')

TAPE: REAL  (SBUX/SPY total-return monthly, 1993-02..2026-05, 400 months, fp 1d50b4e1e153)

Month       Mean   t-naive     t-HAC     n  Signal?
Jan       +1.26%     +0.61     +0.53    33  noise
Feb       +1.01%     +1.06     +1.21    34  noise
Mar       +2.45%     +1.76     +1.71    34  noise
Apr       +0.09%     +0.05     +0.06    34  noise
May       +1.34%     +0.97     +0.94    34  noise
Jun       +1.59%     +1.06     +0.92    33  noise
Jul       -1.83%     -0.97     -0.86    33  noise
Aug (PSL)    +0.64%     +0.50     +0.54    33  noise
Sep (PSL)    +1.51%     +1.18     +1.20    33  noise
Oct (PSL)    +2.14%     +1.52     +1.34    33  noise
Nov (PSL)    +0.72%     +0.49     +0.49    33  noise
Dec       -0.46%     -0.41     -0.60    33  noise


> 💡 In plain words: not one month clears |t| ≥ 2 on either statistic. The largest excess (March, t ≈ 1.71 HAC) is off-thesis, and the PSL-launch month August is one of the *weakest* season months (t = 0.54). **H₁ rejected.**

### 4.2 Season vs off-season spread + block-bootstrap CI (H₂, H₃)

In [3]:
print(f'Season (Aug-Nov): {se["season_mean"]*100:+.2f}%/mo  n={se["n_season"]}')
print(f'Off-season:       {se["off_mean"]*100:+.2f}%/mo  n={se["n_off"]}')
print(f'Spread: {se["spread"]*100:+.2f}%/mo  Welch t={se["tstat"]:.2f}')
ci = st.spread_bootstrap_ci(excess, n_boot=5000, seed=724)
print(f'Block-bootstrap 95% CI on spread: [{ci["lo"]*100:.2f}%, {ci["hi"]*100:.2f}%]  '
      f'(point {ci["point"]*100:+.2f}%, n_boot={ci["n_boot"]})')
print('CI straddles 0:', ci['lo'] < 0 < ci['hi'])

Season (Aug-Nov): +1.25%/mo  n=132
Off-season:       +0.69%/mo  n=268
Spread: +0.56%/mo  Welch t=0.65


Block-bootstrap 95% CI on spread: [-0.84%, 1.92%]  (point +0.56%, n_boot=5000)
CI straddles 0: True


> 💡 In plain words: the Welch t is 0.65 and the bootstrap CI runs roughly −0.8% to +1.9% — it swamps the 0.56% point estimate. The season spread is noise. **H₂ and H₃ rejected.**

### 4.3 The window placebo — is Aug–Nov special? (H₄)

In [4]:
wp = st.window_placebo(excess)
disp = wp.copy(); disp['spread']=(disp['spread']*100).round(2)
for rk,row in disp.iterrows():
    flag = '  <-- PUMPKIN SPICE' if row['is_psl'] else ''
    print(f"  #{rk+1}: {row['months']:22s} spread {row['spread']:+.2f}%  t={row['tstat']:+.2f}{flag}")
psl_rank=int(wp[wp['is_psl']].index[0])+1; top=wp.iloc[0]
print(f'\nPSL window ranks #{psl_rank} of {len(wp)}; the strongest is {top["months"]} '
      f'({top["spread"]*100:+.2f}%), which has nothing to do with pumpkins.')

  #1: Mar-Apr-May-Jun        spread +0.74%  t=+0.79
  #2: Aug-Sep-Oct-Nov        spread +0.56%  t=+0.65  <-- PUMPKIN SPICE
  #3: Feb-Mar-Apr-May        spread +0.53%  t=+0.58
  #4: Jan-Feb-Mar-Apr        spread +0.49%  t=+0.51
  #5: Dec-Jan-Feb-Mar        spread +0.30%  t=+0.33
  #6: Sep-Oct-Nov-Dec        spread +0.15%  t=+0.18
  #7: Oct-Nov-Dec-Jan        spread +0.06%  t=+0.06
  #8: Nov-Dec-Jan-Feb        spread -0.36%  t=-0.40
  #9: Jul-Aug-Sep-Oct        spread -0.39%  t=-0.42
  #10: Jun-Jul-Aug-Sep        spread -0.59%  t=-0.64
  #11: May-Jun-Jul-Aug        spread -0.65%  t=-0.70
  #12: Apr-May-Jun-Jul        spread -0.86%  t=-0.88

PSL window ranks #2 of 12; the strongest is Mar-Apr-May-Jun (+0.74%), which has nothing to do with pumpkins.


> 💡 In plain words: a real seasonal edge would put Aug–Nov at the top of the placebo ranking. It lands **#2**, behind an off-thesis spring window. When the story-window isn't the winning window, the story is decoration. **H₄ rejected.**

### 4.4 Rotation race + the survivorship trap (H₅a)

In [5]:
res = {
    'seasonal rotation (SBUX Aug-Nov, else SPY)': st.summary(rot, rf=rf),
    'rotation net 5bp':                           st.summary(rot_net, rf=rf),
    'market-neutral pair, season only':           st.summary(timer, rf=rf),
    'market-neutral pair, ALL YEAR':              st.summary(st.buy_hold(excess)),
    'buy & hold SPY':                             st.summary(bh_spy, rf=rf),
    'buy & hold SBUX':                            st.summary(bh_sbux, rf=rf),
}
display(pd.DataFrame(res).T[['cagr','sharpe','vol_ann','max_drawdown','n']].round(3))
print('The season-only pair (Sharpe ~0.27) is WORSE than holding the pair all year (~0.35):')
print('the pumpkin window does not concentrate SBUX alpha, it dilutes it. The rotation only')
print('edges SPY by borrowing SBUX beta 4 months/yr -- pure single-name survivorship.')

,cagr,sharpe,vol_ann,max_drawdown,n
"seasonal rotation (SBUX Aug-Nov, else SPY)",0.151,0.659,0.212,-0.635,400.0
rotation net 5bp,0.150,0.654,0.212,-0.636,400.0
"market-neutral pair, season only",0.056,0.269,0.154,-0.349,400.0
"market-neutral pair, ALL YEAR",0.062,0.350,0.299,-0.677,400.0
buy & hold SPY,0.109,0.610,0.148,-0.508,400.0
buy & hold SBUX,0.176,0.582,0.335,-0.764,400.0


The season-only pair (Sharpe ~0.27) is WORSE than holding the pair all year (~0.35):
the pumpkin window does not concentrate SBUX alpha, it dilutes it. The rotation only
edges SPY by borrowing SBUX beta 4 months/yr -- pure single-name survivorship.


> 💡 In plain words: the market-neutral SBUX-vs-SPY pair earns a *higher* Sharpe when held all year (0.35) than when held only in pumpkin-spice season (0.27). If the season were where the alpha lived, restricting to it would *raise* the Sharpe. It lowers it. **H₅a rejected.**

### 4.5 Sub-period stability (H₅b)

In [6]:
d.index = pd.DatetimeIndex(d.index)
for lab, yr in [('1993-2009',(1993,2009)), ('2010-on',(2010,2030))]:
    sl = d[(d.index.year>=yr[0]) & (d.index.year<=yr[1])]
    r = st.season_tstat(sl['excess'])
    print(f'{lab}: season={r["season_mean"]*100:+.2f}%  off={r["off_mean"]*100:+.2f}%  '
          f'spread={r["spread"]*100:+.2f}%  Welch t={r["tstat"]:.2f}  n_on={r["n_season"]}')
print('\nThe season spread FLIPS SIGN across halves -> the +2.06 in 2010-on is a snooped split,')
print('not a stable law; the full-sample t=0.65 is the honest headline.')

1993-2009: season=+0.99%  off=+1.67%  spread=-0.69%  Welch t=-0.47  n_on=68
2010-on: season=+1.53%  off=-0.31%  spread=+1.84%  Welch t=2.06  n_on=64

The season spread FLIPS SIGN across halves -> the +2.06 in 2010-on is a snooped split,
not a stable law; the full-sample t=0.65 is the honest headline.


> 💡 In plain words: the season spread is *negative* in 1993–2009 (t = −0.47) and only nominally positive 2010-on (t = 2.06 — but on a split *we chose*, over half the sample, off a full-sample t = 0.65). Per the desk's inference bar, a snooped, sign-flipping half reads **WEAK at most**, never REAL. **H₅b rejected.**

### 4.6 Robustness — the QSR basket

In [7]:
try:
    b = repro.as_of(data.fetch_basket())
    if b.empty: raise RuntimeError('basket cache miss')
    rb = st.season_tstat(b['excess'])
    print(f'QSR basket (SBUX/MCD/YUM/CMG) excess over SPY, {b.index.min().date()}..{b.index.max().date()} ({len(b)} months):')
    print(f'  season {rb["season_mean"]*100:+.2f}%  off {rb["off_mean"]*100:+.2f}%  '
          f'spread {rb["spread"]*100:+.2f}%  Welch t={rb["tstat"]:.2f}')
except Exception as e:
    print('basket leg needs the cached parquet (examples/verify.py --fetch):', e)

QSR basket (SBUX/MCD/YUM/CMG) excess over SPY, 2006-02-28..2026-05-31 (244 months):
  season +0.71%  off +0.28%  spread +0.43%  Welch t=0.71


> 💡 In plain words: broadening from Starbucks to a coffee/QSR basket doesn't rescue the effect — the season spread is +0.43%/mo at t = 0.71, still noise. Whatever autumn pop the story imagines is neither a Starbucks quirk nor a broad QSR law; it isn't there.

## 5 · The verdict

H₁ rejected (no significant month). H₂/H₃ rejected (spread t = 0.65, CI [−0.84%, +1.92%]). H₄ rejected (PSL window ranks #2 of 12). H₅a rejected (season-only pair 0.27 < all-year pair 0.35). H₅b rejected (sign-flipping sub-periods). → Signal `NONE`, Tradability `MIRAGE`, pumpkin premium `BUSTED`.

## 6 · Could you trade it?

No — not as a seasonal edge. The long-only rotation's Sharpe 0.66 vs SPY's 0.61 is entirely *borrowed SBUX beta* dropped into four months, and it's not even the right four (Mar–Jun beats it in the placebo). The market-neutral seasonal pair — the honest isolation of "the season" — is Sharpe 0.27 gross, 0.26 net, and *below* the all-year pair. There is no seasonal alpha to defend, so the break-even-cost question is moot. What's real here is survivorship: you'd have needed to know, in 1993, that SBUX would be one of the great stocks of the era.

## 7 · Going further

Forks: (a) key the test to SBUX's **autumn-quarter same-store sales** — is even the *fundamental* pumpkin bump real, or does the fall quarter just look like the rest of a growth story? (b) an event-study around each year's **PSL launch date** (a few trading days), where a marketing pop, if any, would actually live — not a four-month window; (c) cross-check against [307 Coffee-Seasonality](../../307-coffee-seasonality/) (the bean, not the brand) and the calendar-effect family. Companion caveat: any "buy the beloved brand" seasonal is one survivor away from a data-mined mirage.